# Zero-shot CIFAR-10 classification with CLIP

This walkthrough turns a compact experiment into a reproducible inference workflow. It builds a CIFAR-10 classifier from natural-language prompts, evaluates cosine-similarity logits, and inspects a portable prediction artifact.

The checked-in metrics come from the original executed experiment. Remote model and dataset downloads are disabled by default so the notebook remains safe to open and render offline.

## Method

For every class, 18 prompt variants are embedded with CLIP's text encoder. Each prompt embedding is normalized, the class embeddings are averaged, and the result is normalized again. Image embeddings are normalized before multiplying them by the class-weight matrix. The resulting scores are scaled cosine similarities—no CIFAR-10 labels are used to train the classifier.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from cifar10_clip.prompts import CIFAR10_CLASSES, CIFAR10_TEMPLATES, format_prompts

RUN_EVALUATION = False
len(CIFAR10_CLASSES), len(CIFAR10_TEMPLATES)

## Prompt ensemble

The templates vary image quality, size, contrast, and article choice. Preserving template order makes the classifier construction deterministic.

In [ ]:
cat_prompts = format_prompts("cat", CIFAR10_TEMPLATES)
cat_prompts[:5]

## Optional end-to-end evaluation

The command below loads OpenAI CLIP `ViT-B/32`, downloads the CIFAR-10 test split, builds the prompt ensemble, and writes fresh metrics and predictions. Set the flag only in an environment with network access and enough disk space.

In [ ]:
if RUN_EVALUATION:
    from cifar10_clip.cli import main

    main([
        "--model", "ViT-B/32",
        "--device", "auto",
        "--data-dir", str(ROOT / "data"),
        "--output-dir", str(ROOT / "runs" / "clip-evaluation"),
    ])

## Historical result

These values are provenance-labeled snapshots from the original executed notebook, not a claim that this notebook reran the model during rendering.

In [ ]:
metrics = json.loads((ROOT / "artifacts" / "metrics.json").read_text())
pd.Series(metrics, name="value")

## Portable prediction artifact

The original predictions were stored in a Python pickle. The repository publishes a deterministic CSV instead, with one row per CIFAR-10 test image and both the numeric and human-readable predicted class. The conversion script uses a restricted unpickler and refuses global object lookups.

In [ ]:
predictions = pd.read_csv(ROOT / "artifacts" / "zero_shot_predictions.csv")
display(predictions.head())
display(predictions["predicted_class_name"].value_counts().sort_index())

## Limitations and next steps

- Zero-shot performance depends on prompt wording and the pretraining distribution.
- CIFAR-10 is small and coarse-grained; the result does not establish robustness on larger datasets.
- The historical notebook reported top-1 and top-5 accuracy but did not include calibration or per-class recall.

Useful extensions would compare prompt sets, measure class-level error patterns, add confidence calibration, and evaluate newer open vision-language encoders under the same artifact contract.